In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/adaptive-lora/checkpoints/baseline', exist_ok=True)
os.makedirs('/content/drive/MyDrive/adaptive-lora/results/tables', exist_ok=True)
print("Drive ready")

Mounted at /content/drive
Drive ready


In [5]:
%%capture
!pip install transformers peft accelerate bitsandbytes datasets wandb trl

In [6]:
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from datasets import load_dataset
from torch.utils.data import DataLoader

print("Imports done")

Imports done


In [7]:
model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Loaded. Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded. Memory used: 4.13 GB


In [8]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,                    # uniform rank, every layer
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


In [9]:
dataset = load_dataset("fancyzhx/ag_news")

def format_example(example):
    label_names = ['World', 'Sports', 'Business', 'Sci/Tech']
    label = label_names[example['label']]
    text = f"""Classify the following news article into one of these categories: World, Sports, Business, Sci/Tech.

Article: {example['text']}

Category: {label}"""
    return {"text": text}

formatted_train = dataset['train'].map(format_example)
formatted_test = dataset['test'].map(format_example)

print(f"Train: {len(formatted_train)}")
print(f"Test:  {len(formatted_test)}")

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Train: 120000
Test:  7600


In [10]:
# 8000 training examples is enough to see real learning,
# while keeping training time reasonable on T4 free tier
TRAIN_SIZE = 8000
TEST_SIZE = 1000   # subset of test set for faster evaluation

train_subset = formatted_train.shuffle(seed=42).select(range(TRAIN_SIZE))
test_subset = formatted_test.shuffle(seed=42).select(range(TEST_SIZE))

print(f"Using {len(train_subset)} train examples")
print(f"Using {len(test_subset)} test examples")

Using 8000 train examples
Using 1000 test examples


In [11]:
MAX_LENGTH = 256

def tokenize(example):
    result = tokenizer(
        example['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized_train = train_subset.map(tokenize, remove_columns=train_subset.column_names)
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Tokenized {len(tokenized_train)} train examples")

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenized 8000 train examples


In [12]:
train_loader = DataLoader(tokenized_train, batch_size=4, shuffle=True)
print(f"Batches per epoch: {len(train_loader)}")

EPOCHS = 2
total_steps = len(train_loader) * EPOCHS
print(f"Total training steps: {total_steps}")

Batches per epoch: 2000
Total training steps: 4000


In [13]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4
)

CHECKPOINT_EVERY = 500  # save every 500 steps, not just at the end
SAVE_PATH = '/content/drive/MyDrive/adaptive-lora/checkpoints/baseline'

model.train()
step = 0
losses = []

for epoch in range(EPOCHS):
    print(f"\n=== EPOCH {epoch+1}/{EPOCHS} ===")
    for batch in tqdm(train_loader):
        batch = {k: v.to(model.device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        losses.append(loss.item())
        step += 1

        if step % 100 == 0:
            avg_loss = np.mean(losses[-100:])
            print(f"Step {step}/{total_steps} | Avg Loss (last 100): {avg_loss:.4f}")

        if step % CHECKPOINT_EVERY == 0:
            model.save_pretrained(SAVE_PATH)
            print(f"  Checkpoint saved at step {step}")

# Final save
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"\nTraining complete. Final model saved to {SAVE_PATH}")

# Save loss history
with open('/content/drive/MyDrive/adaptive-lora/results/baseline_losses.json', 'w') as f:
    json.dump(losses, f)


=== EPOCH 1/2 ===


  0%|          | 0/2000 [00:00<?, ?it/s][transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
  5%|▌         | 100/2000 [06:49<2:10:05,  4.11s/it]

Step 100/4000 | Avg Loss (last 100): 0.6113


 10%|█         | 200/2000 [13:42<2:03:20,  4.11s/it]

Step 200/4000 | Avg Loss (last 100): 0.5105


...

100%|█████████▉| 1999/2000 [2:17:38<00:04,  4.12s/it]

Step 4000/4000 | Avg Loss (last 100): 0.4348


100%|██████████| 2000/2000 [2:17:42<00:00,  4.13s/it]

  Checkpoint saved at step 4000



Training complete. Final model saved to /content/drive/MyDrive/adaptive-lora/checkpoints/baseline
